# TT-60: Order Event Pipeline Validation

Functional validation of Order and ComplexOrder event handling against the TastyTrade **sandbox**.

**Phases:**
1. Dry-run equity limit order — validate REST response parses into `PlacedOrder`
2. Dry-run vertical spread — validate multi-leg parsing with `OrderLeg`
3. Start `AccountStreamer` — confirm ORDER and COMPLEX_ORDER queues exist
4. Submit live order + capture streamer event — compare REST vs streamer
5. Cancel order + capture cancellation event — validate status transitions
6. Complex order (OCO) + capture event — validate `PlacedComplexOrder` parsing
7. Model gap analysis — flag fields present in wire but missing from model

In [ ]:
import asyncio
import json
import logging
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from tastytrade.accounts.models import (
    PlacedComplexOrder,
    PlacedOrder,
)
from tastytrade.accounts.streamer import AccountStreamer
from tastytrade.config import RedisConfigManager
from tastytrade.config.enumerations import AccountEventType
from tastytrade.connections import Credentials
from tastytrade.connections.requests import AsyncSessionHandler

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 80)

load_dotenv("/workspace/.env", override=True)

if os.getenv("GRAFANA_CLOUD_TOKEN"):
    from tastytrade.common.observability import init_observability
    init_observability()
    print("Observability: Grafana Cloud enabled")
else:
    logging.basicConfig(level=logging.INFO)
    print("Observability: Local only (no GRAFANA_CLOUD_TOKEN)")

_OBFUSCATE = os.getenv("OBFUSCATE_ACCOUNTS", "false").lower() == "true"


def mask(account_number: str) -> str:
    """Mask account number for safe display, showing only last 4 chars."""
    if not _OBFUSCATE or len(account_number) <= 4:
        return account_number
    return "***" + account_number[-4:]


def mask_json(text: str, account_numbers: list[str]) -> str:
    """Mask all known account numbers in a raw JSON string."""
    if not _OBFUSCATE:
        return text
    for acct in account_numbers:
        text = text.replace(acct, mask(acct))
    return text


print(f"Account obfuscation: {'ON' if _OBFUSCATE else 'OFF'}")

# 1. Authenticate & Connect (Sandbox)

In [ ]:
config = RedisConfigManager(env_file="/workspace/.env")
config.initialize(force=True)

credentials = Credentials(config=config, env="Test")
session = await AsyncSessionHandler.create(credentials)

session_token = session.session.headers["Authorization"]
base_url = session.base_url
account_number = credentials.account_number

print(f"Environment: {'Sandbox' if credentials.is_sandbox else 'Production'}")
print(f"Base URL: {base_url}")
print(f"Account: {mask(account_number)}")

assert credentials.is_sandbox, "SAFETY: This notebook must run against sandbox only!"

# 2. Helper Functions

REST wrappers for order submission, dry-run, and cancellation.

In [ ]:
# Capture raw payloads for model gap analysis
captured_rest_payloads: list[dict] = []
captured_streamer_payloads: list[dict] = []


async def dry_run_order(acct: str, order_body: dict) -> dict:
    """POST /accounts/{acct}/orders/dry-run — preview without submitting."""
    url = f"{base_url}/accounts/{acct}/orders/dry-run"
    async with session.session.post(url, json=order_body) as resp:
        data = await resp.json()
        print(f"Dry-run status: {resp.status}")
        print(mask_json(json.dumps(data, indent=2, default=str)[:3000], [acct]))
        return data


async def submit_order(acct: str, order_body: dict) -> dict:
    """POST /accounts/{acct}/orders — submit a live order."""
    url = f"{base_url}/accounts/{acct}/orders"
    async with session.session.post(url, json=order_body) as resp:
        data = await resp.json()
        print(f"Submit status: {resp.status}")
        order_data = data.get("data", {}).get("order", data.get("data", {}))
        if order_data:
            captured_rest_payloads.append(order_data)
        print(mask_json(json.dumps(data, indent=2, default=str)[:3000], [acct]))
        return data


async def cancel_order(acct: str, order_id: int) -> dict:
    """DELETE /accounts/{acct}/orders/{id} — cancel an open order."""
    url = f"{base_url}/accounts/{acct}/orders/{order_id}"
    async with session.session.delete(url) as resp:
        data = await resp.json()
        print(f"Cancel status: {resp.status}")
        print(mask_json(json.dumps(data, indent=2, default=str)[:2000], [acct]))
        return data


async def submit_complex_order(acct: str, body: dict) -> dict:
    """POST /accounts/{acct}/complex-orders — submit a complex order."""
    url = f"{base_url}/accounts/{acct}/complex-orders"
    async with session.session.post(url, json=body) as resp:
        data = await resp.json()
        print(f"Complex order status: {resp.status}")
        complex_data = data.get("data", {})
        if complex_data:
            captured_rest_payloads.append(complex_data)
        print(mask_json(json.dumps(data, indent=2, default=str)[:3000], [acct]))
        return data


async def cancel_complex_order(acct: str, order_id: int) -> dict:
    """DELETE /accounts/{acct}/complex-orders/{id} — cancel a complex order."""
    url = f"{base_url}/accounts/{acct}/complex-orders/{order_id}"
    async with session.session.delete(url) as resp:
        data = await resp.json()
        print(f"Cancel complex status: {resp.status}")
        print(mask_json(json.dumps(data, indent=2, default=str)[:2000], [acct]))
        return data


print("Helper functions defined.")

# Phase 1: Dry-Run — Equity Limit Order

Submit a simple equity limit order (Buy 1 SPY @ $1.00 — won't fill) to the dry-run endpoint.
Validate the response parses into `PlacedOrder`.

In [ ]:
equity_order_body = {
    "time-in-force": "Day",
    "order-type": "Limit",
    "price": "1.00",
    "price-effect": "Debit",
    "legs": [
        {
            "instrument-type": "Equity",
            "symbol": "SPY",
            "action": "Buy to Open",
            "quantity": "1",
        }
    ],
}

display(Markdown("### Dry-Run: Buy 1 SPY @ $1.00 (Limit)"))
dry_run_result = await dry_run_order(account_number, equity_order_body)

# Validate parsing into PlacedOrder
order_data = dry_run_result.get("data", {}).get("order", {})
if order_data:
    parsed = PlacedOrder.model_validate(order_data)
    display(Markdown("**PlacedOrder parse: SUCCESS**"))
    print(f"  id={parsed.id}, status={parsed.status.value}, type={parsed.order_type.value}")
    print(f"  legs={len(parsed.legs)}, symbol={parsed.legs[0].symbol if parsed.legs else 'N/A'}")
    print(f"  price={parsed.price}, price_effect={parsed.price_effect}")
else:
    display(Markdown("**No order data in dry-run response — check response format above**"))

# Phase 2: Dry-Run — Vertical Spread (2-Leg)

Submit a vertical call spread on SPY to test multi-leg parsing.
Validate `OrderLeg` action enums and instrument types.

In [ ]:
# Use far-OTM strikes so the order is clearly not fillable
# Adjust symbols to match current sandbox option chain if needed
spread_order_body = {
    "time-in-force": "Day",
    "order-type": "Limit",
    "price": "0.01",
    "price-effect": "Debit",
    "legs": [
        {
            "instrument-type": "Equity Option",
            "symbol": "SPY   260320C00800000",
            "action": "Buy to Open",
            "quantity": "1",
        },
        {
            "instrument-type": "Equity Option",
            "symbol": "SPY   260320C00810000",
            "action": "Sell to Open",
            "quantity": "1",
        },
    ],
}

display(Markdown("### Dry-Run: SPY 800/810 Call Spread @ $0.01 Debit"))
spread_result = await dry_run_order(account_number, spread_order_body)

# Validate parsing
spread_data = spread_result.get("data", {}).get("order", {})
if spread_data:
    parsed_spread = PlacedOrder.model_validate(spread_data)
    display(Markdown("**PlacedOrder parse (spread): SUCCESS**"))
    print(f"  id={parsed_spread.id}, legs={len(parsed_spread.legs)}")
    for i, leg in enumerate(parsed_spread.legs):
        print(f"  Leg {i}: {leg.action.value} {leg.quantity} {leg.symbol} ({leg.instrument_type.value})")
else:
    display(Markdown("**No order data — check response format above**"))

# Phase 3: Start AccountStreamer

Create and start the `AccountStreamer`. Confirm ORDER and COMPLEX_ORDER queues are initialized.

In [ ]:
# Reset singleton so we get a fresh instance
AccountStreamer.instance = None

streamer = AccountStreamer(credentials)
await streamer.start()

print("AccountStreamer started.")
print(f"Queues available:")
for event_type in AccountEventType:
    q = streamer.queues[event_type]
    print(f"  {event_type.value}: {q.qsize()} items (hydrated)")

assert AccountEventType.ORDER in streamer.queues, "ORDER queue missing!"
assert AccountEventType.COMPLEX_ORDER in streamer.queues, "COMPLEX_ORDER queue missing!"
display(Markdown("**ORDER and COMPLEX_ORDER queues confirmed.**"))

# Phase 4: Submit Live Order + Capture Streamer Event

Submit a far-OTM equity limit order (won't fill), then listen for the Order event on the streamer.
Compare REST response fields vs streamer event fields.

In [ ]:
live_order_body = {
    "time-in-force": "Day",
    "order-type": "Limit",
    "price": "1.00",
    "price-effect": "Debit",
    "legs": [
        {
            "instrument-type": "Equity",
            "symbol": "SPY",
            "action": "Buy to Open",
            "quantity": "1",
        }
    ],
}

display(Markdown("### Submitting live order: Buy 1 SPY @ $1.00 (Limit)"))
live_result = await submit_order(account_number, live_order_body)

rest_order_data = live_result.get("data", {}).get("order", {})
live_order_id = rest_order_data.get("id")
print(f"\nOrder ID from REST: {live_order_id}")

# Parse REST response
if rest_order_data:
    rest_parsed = PlacedOrder.model_validate(rest_order_data)
    print(f"REST parse: id={rest_parsed.id}, status={rest_parsed.status.value}")

# Listen for streamer event
display(Markdown("### Listening for Order event on streamer..."))
order_queue = streamer.queues[AccountEventType.ORDER]

streamer_events = []
try:
    # May receive multiple status updates (Received -> Routed -> Live)
    for _ in range(5):
        event = await asyncio.wait_for(order_queue.get(), timeout=10.0)
        streamer_events.append(event)
        print(f"  Streamer event: id={event.id}, status={event.status.value}")
        if event.status.value in ("Live", "Rejected", "Cancelled"):
            break
except asyncio.TimeoutError:
    print(f"  (timeout after {len(streamer_events)} events)")

if streamer_events:
    display(Markdown(f"**Captured {len(streamer_events)} streamer events for order {live_order_id}**"))
    last_event = streamer_events[-1]
    print(f"Final status: {last_event.status.value}")
    print(f"Type: {type(last_event).__name__}")
    assert isinstance(last_event, PlacedOrder), f"Expected PlacedOrder, got {type(last_event)}"
else:
    display(Markdown("**No streamer events received — check connection.**"))

# Phase 5: Cancel Order + Capture Cancellation Event

Cancel the order from Phase 4 and observe the status transition on the streamer.

In [ ]:
if live_order_id:
    display(Markdown(f"### Cancelling order {live_order_id}"))
    cancel_result = await cancel_order(account_number, live_order_id)

    # Listen for cancellation events on streamer
    display(Markdown("### Listening for cancellation events..."))
    cancel_events = []
    try:
        for _ in range(5):
            event = await asyncio.wait_for(order_queue.get(), timeout=10.0)
            cancel_events.append(event)
            print(f"  Streamer event: id={event.id}, status={event.status.value}")
            if event.status.value in ("Cancelled", "Rejected"):
                break
    except asyncio.TimeoutError:
        print(f"  (timeout after {len(cancel_events)} events)")

    if cancel_events:
        statuses = [e.status.value for e in cancel_events]
        display(Markdown(f"**Status transitions:** {' -> '.join(statuses)}"))
        assert cancel_events[-1].status.value == "Cancelled", \
            f"Expected Cancelled, got {cancel_events[-1].status.value}"
        display(Markdown("**Cancellation confirmed.**"))
    else:
        display(Markdown("**No cancellation events — check streamer connection.**"))
else:
    display(Markdown("**No order to cancel — Phase 4 did not produce an order ID.**"))

# Phase 6: Complex Order (OCO) + Capture Event

Submit an OCO (One-Cancels-Other) complex order via `/complex-orders`.
Validate `PlacedComplexOrder` parsing with nested `PlacedOrder` objects.

In [ ]:
# OCO: two limit orders — only one can fill (both far OTM so neither will)
oco_body = {
    "type": "OCO",
    "orders": [
        {
            "time-in-force": "Day",
            "order-type": "Limit",
            "price": "1.00",
            "price-effect": "Debit",
            "legs": [
                {
                    "instrument-type": "Equity",
                    "symbol": "SPY",
                    "action": "Buy to Open",
                    "quantity": "1",
                }
            ],
        },
        {
            "time-in-force": "Day",
            "order-type": "Limit",
            "price": "9999.00",
            "price-effect": "Credit",
            "legs": [
                {
                    "instrument-type": "Equity",
                    "symbol": "SPY",
                    "action": "Sell to Close",
                    "quantity": "1",
                }
            ],
        },
    ],
}

display(Markdown("### Submitting OCO complex order"))
complex_result = await submit_complex_order(account_number, oco_body)

complex_data = complex_result.get("data", {})
complex_order_id = complex_data.get("id")
print(f"\nComplex order ID: {complex_order_id}")

# Parse REST response into PlacedComplexOrder
if complex_data and "id" in complex_data:
    complex_parsed = PlacedComplexOrder.model_validate(complex_data)
    display(Markdown("**PlacedComplexOrder parse: SUCCESS**"))
    print(f"  type={complex_parsed.type.value}, sub-orders={len(complex_parsed.orders)}")
    for i, sub in enumerate(complex_parsed.orders):
        print(f"  Sub-order {i}: id={sub.id}, status={sub.status.value}, type={sub.order_type.value}")

# Listen for ComplexOrder event on streamer
display(Markdown("### Listening for ComplexOrder streamer events..."))
complex_queue = streamer.queues[AccountEventType.COMPLEX_ORDER]

# Also drain any Order events that come alongside
complex_events = []
order_side_events = []
try:
    for _ in range(10):
        # Check both queues
        done, pending = await asyncio.wait(
            [
                asyncio.create_task(complex_queue.get()),
                asyncio.create_task(order_queue.get()),
            ],
            timeout=10.0,
            return_when=asyncio.FIRST_COMPLETED,
        )
        for task in pending:
            task.cancel()
        if not done:
            break
        for task in done:
            event = task.result()
            if isinstance(event, PlacedComplexOrder):
                complex_events.append(event)
                print(f"  COMPLEX_ORDER: id={event.id}, type={event.type.value}, sub-orders={len(event.orders)}")
            elif isinstance(event, PlacedOrder):
                order_side_events.append(event)
                print(f"  ORDER (sub): id={event.id}, status={event.status.value}")
except asyncio.TimeoutError:
    pass

display(Markdown(f"**Captured {len(complex_events)} ComplexOrder + {len(order_side_events)} Order events**"))

# Cancel the complex order
if complex_order_id:
    display(Markdown(f"### Cancelling complex order {complex_order_id}"))
    await cancel_complex_order(account_number, complex_order_id)

    # Drain terminal events
    try:
        for _ in range(5):
            done, pending = await asyncio.wait(
                [
                    asyncio.create_task(complex_queue.get()),
                    asyncio.create_task(order_queue.get()),
                ],
                timeout=10.0,
                return_when=asyncio.FIRST_COMPLETED,
            )
            for task in pending:
                task.cancel()
            if not done:
                break
            for task in done:
                event = task.result()
                if isinstance(event, PlacedComplexOrder):
                    print(f"  COMPLEX_ORDER terminal: id={event.id}, terminal_at={event.terminal_at}")
                elif isinstance(event, PlacedOrder):
                    print(f"  ORDER terminal: id={event.id}, status={event.status.value}")
    except asyncio.TimeoutError:
        pass

    display(Markdown("**Complex order cancelled.**"))

# Phase 7: Model Gap Analysis

Compare wire payload fields vs Pydantic model fields.
Flag fields present in the wire payload but missing from our models (`extra="ignore"` swallows them).

In [ ]:
def analyze_model_gaps(payload: dict, model_cls: type, label: str) -> dict:
    """Compare wire payload keys against a Pydantic model's defined fields."""
    # Get model field aliases (wire names)
    model_aliases = set()
    model_fields_by_alias: dict[str, str] = {}
    for field_name, field_info in model_cls.model_fields.items():
        alias = field_info.alias or field_name
        model_aliases.add(alias)
        model_fields_by_alias[alias] = field_name

    wire_keys = set(payload.keys())

    missing_from_model = wire_keys - model_aliases
    missing_from_wire = model_aliases - wire_keys

    # Fields that came back null when we might expect values
    null_fields = [
        model_fields_by_alias.get(k, k)
        for k in wire_keys & model_aliases
        if payload.get(k) is None
    ]

    result = {
        "label": label,
        "wire_fields": len(wire_keys),
        "model_fields": len(model_aliases),
        "missing_from_model": sorted(missing_from_model),
        "missing_from_wire": sorted(missing_from_wire),
        "null_fields": sorted(null_fields),
    }

    display(Markdown(f"### {label}"))
    print(f"Wire fields: {result['wire_fields']}, Model fields: {result['model_fields']}")

    if missing_from_model:
        display(Markdown(f"**Fields in wire but NOT in model (swallowed by extra=ignore):**"))
        for f in sorted(missing_from_model):
            sample_val = str(payload.get(f, ""))[:60]
            print(f"  {f}: {sample_val}")
    else:
        display(Markdown("**No missing fields — model covers all wire fields.**"))

    if null_fields:
        print(f"\nNull fields ({len(null_fields)}): {', '.join(null_fields[:10])}")

    return result


# Analyze captured REST payloads
display(Markdown("## Gap Analysis Results"))
gap_results = []

for i, payload in enumerate(captured_rest_payloads):
    # Detect whether it's a complex order or regular order
    if "orders" in payload and "type" in payload:
        result = analyze_model_gaps(payload, PlacedComplexOrder, f"REST payload #{i+1} (ComplexOrder)")
        gap_results.append(result)
        # Also analyze nested orders
        for j, sub in enumerate(payload.get("orders", [])):
            sub_result = analyze_model_gaps(sub, PlacedOrder, f"  Sub-order #{j+1}")
            gap_results.append(sub_result)
    elif "legs" in payload or "order-type" in payload:
        result = analyze_model_gaps(payload, PlacedOrder, f"REST payload #{i+1} (PlacedOrder)")
        gap_results.append(result)

# Summary table
if gap_results:
    display(Markdown("## Summary"))
    summary_data = [{
        "Payload": r["label"],
        "Wire": r["wire_fields"],
        "Model": r["model_fields"],
        "Missing from Model": len(r["missing_from_model"]),
        "Null Fields": len(r["null_fields"]),
    } for r in gap_results]
    display(pd.DataFrame(summary_data))

    total_missing = sum(len(r["missing_from_model"]) for r in gap_results)
    if total_missing == 0:
        display(Markdown("**All wire fields are covered by our models.**"))
    else:
        display(Markdown(f"**{total_missing} total fields missing from models — review for promotion.**"))
else:
    display(Markdown("*No payloads captured for analysis. Run Phases 4-6 first.*"))

# Cleanup

In [ ]:
await streamer.close()
await session.close()

if os.getenv("GRAFANA_CLOUD_TOKEN"):
    from tastytrade.common.observability import shutdown_observability
    shutdown_observability()

print("All connections closed.")